**Finesse — Gamified Budgeting & Financial Health Scorer**

Anggota:
1. CFCC319D6Y0190 - Patrick Nicxon Hutabarat - Full-Stack Web Developer
2. CDCC319D6X0998 - Dame Theresia Rejeki Sidauruk - Data Science
3. CDCC319D6X1254 - Cikita Natasya Br Sembiring - Data Scientist
4. CACC319D6Y0343 - Rayza Indafri Yahya - AI Engineer
5. CACC319D6Y1720 - Samuel Gautama Manik - AI Engineer

## **Langkah 1: Persiapan Library dan Memuat Data**
Pada langkah pertama ini, dipanggil library dasar yang dibutuhkan, yaitu pandas untuk manipulasi data tabel dan numpy untuk perhitungan matematis. Selanjutnya, dataset hasil wrangling (finesse_dataset.csv) dimuat ke dalam memori agar siap diproses lebih lanjut.


In [ ]:
# Import Library dan Load Dataset
import pandas as pd
import numpy as np

# Membaca data yang sudah bersih
df = pd.read_csv('finesse_dataset.csv')

# Menampilkan 3 baris awal untuk memastikan data terbaca
print("Data berhasil dimuat!")
display(df.head(3))

Data berhasil dimuat!


,transaction_id,user_id,date,amount,category,payment_method,monthly_budget,cumulative_spend,financial_health_score
0,TRANS033951,CUST00001,2023-03-14,179000,Hiburan & Nongkrong,Bank Transfer,1614000,179000,88.91
1,TRANS023682,CUST00001,2023-03-24,29000,Makan & Minum,Bank Transfer,1614000,208000,87.11
2,TRANS002749,CUST00001,2023-04-14,480000,Belanja Umum,Credit Card,1614000,480000,70.26


## **Langkah 2: Ekstraksi Fitur Berbasis Waktu (Temporal Features)**
Di langkah kedua ini, dilakukan ekstraksi fitur waktu. Model tidak memahami format tanggal seperti "2024-05-29". Oleh karena itu, tanggal transaksi dipecah untuk menarik pola kebiasaan, seperti apakah transaksi terjadi pada akhir pekan (hari libur) atau pada akhir bulan (saat anggaran mulai menipis).

In [ ]:
# Mengubah tipe data teks menjadi format datetime yang bisa diolah
df['date'] = pd.to_datetime(df['date'])

# Mengekstrak urutan hari (0 = Senin, 6 = Minggu)
df['day_of_week'] = df['date'].dt.dayofweek

# Menambahkan fitur biner: bernilai 1 jika akhir pekan (Sabtu/Minggu), 0 jika hari kerja
df['is_weekend'] = df['day_of_week'].apply(lambda x: 1 if x >= 5 else 0)

# Menambahkan fitur biner: bernilai 1 jika transaksi terjadi di akhir bulan, 0 jika bukan
df['is_month_end'] = df['date'].dt.is_month_end.astype(int)

print("Fitur waktu berhasil ditambahkan!")

Fitur waktu berhasil ditambahkan!


## **Langkah 3: Ekstraksi Fitur Rasio Finansial**
Langkah ketiga berfokus pada penciptaan rasio finansial. Mengingat arah proyek ini adalah fintech, rasio ini berfungsi sebagai pengukur tingkat keborosan. Di sini, dihitung persentase besaran satu transaksi terhadap anggaran bulanan, serta seberapa dekat total pengeluaran dengan batas anggaran.

In [ ]:
# Menghitung seberapa besar 1 transaksi mengonsumsi anggaran bulanan
df['transaction_to_budget_ratio'] = df['amount'] / df['monthly_budget']

# Menghitung utilitas anggaran (seberapa dekat pengguna dengan batas maksimal budget)
df['budget_utilization_ratio'] = df['cumulative_spend'] / df['monthly_budget']

print("Fitur rasio finansial berhasil dihitung!")

Fitur rasio finansial berhasil dihitung!


## **Langkah 4: Pembuatan Profil Kebiasaan Pengguna**
Pada sel keempat, dilakukan analisis terhadap profil kebiasaan masing-masing individu. Pertama, rata-rata pengeluaran per pengguna dihitung. Setelah itu, nilai transaksi saat ini dibandingkan dengan rata-rata tersebut. Tujuannya adalah untuk mendeteksi apakah suatu pengeluaran tergolong normal atau melampaui kebiasaan harian pengguna tersebut.

In [ ]:
# Menghitung rata-rata nilai transaksi khusus untuk masing-masing user_id
df['user_avg_transaction'] = df.groupby('user_id')['amount'].transform('mean')

# Menghitung selisih antara transaksi saat ini dengan pengeluaran rata-rata pengguna
# Nilai positif berarti transaksi tersebut lebih mahal dari kebiasaan normal
df['amount_vs_user_avg'] = df['amount'] - df['user_avg_transaction']

print("Fitur profil kebiasaan pengguna berhasil dibuat!")

Fitur profil kebiasaan pengguna berhasil dibuat!


## **Langkah 5: Transformasi Data Teks Menjadi Angka (One-Hot Encoding)**
Langkah kelima adalah proses mengubah data teks menjadi angka. Algoritma machine learning tidak dapat menghitung kata seperti "E-Wallet" atau "Makan & Minum". Oleh karena itu, data kategori ini diubah menjadi format biner (0 dan 1). Setelah itu, kolom-kolom lama yang sudah tidak berguna untuk prediksi dibuang agar proses pemodelan berjalan ringan.

In [ ]:
# Mengubah kolom 'category' dan 'payment_method' menjadi kolom angka biner
df = pd.get_dummies(df, columns=['category', 'payment_method'], drop_first=True)

# Menghapus kolom yang sudah diubah atau tidak diperlukan lagi oleh model
kolom_dihapus = ['transaction_id', 'date']
df = df.drop(columns=kolom_dihapus, errors='ignore')

print("Transformasi selesai, kolom teks sudah diubah menjadi angka!")

Transformasi selesai, kolom teks sudah diubah menjadi angka!


## **Langkah 6: Pengecekan Akhir dan Penyimpanan (Export ke CSV)**
Di langkah terakhir, dilakukan pembersihan akhir untuk memastikan tidak ada nilai kosong (missing value) yang muncul akibat proses perhitungan sebelumnya. Setelah aman, dataset yang sudah diperkaya dengan banyak fitur baru ini diekspor menjadi file CSV.

In [ ]:
# Mengisi nilai kosong (NaN) dengan 0 apabila terjadi error pada proses pembagian sebelumnya
df = df.fillna(0)

print("Ukuran dataset final (Baris, Kolom):", df.shape)

# Menyimpan hasil akhir ke file CSV baru
nama_file_baru = 'finesse_dataset_engineered.csv'
df.to_csv(nama_file_baru, index=False)

print(f"Beres! File siap digunakan dan telah disimpan dengan nama: {nama_file_baru}")

# Menampilkan 5 baris pertama dari hasil akhir
display(df.head())

Ukuran dataset final (Baris, Kolom): (15239, 19)
Beres! File siap digunakan dan telah disimpan dengan nama: finesse_dataset_engineered.csv


,user_id,amount,monthly_budget,cumulative_spend,financial_health_score,day_of_week,is_weekend,is_month_end,transaction_to_budget_ratio,budget_utilization_ratio,user_avg_transaction,amount_vs_user_avg,category_Hiburan & Nongkrong,category_Kebutuhan Kuliah,category_Makan & Minum,category_Tagihan & Kos,category_Transportasi,payment_method_Credit Card,payment_method_E-Wallet
0,CUST00001,179000,1614000,179000,88.91,1,0,0,0.110905,0.110905,157045.454545,21954.545455,True,False,False,False,False,False,False
1,CUST00001,29000,1614000,208000,87.11,4,0,0,0.017968,0.128872,157045.454545,-128045.454545,False,False,True,False,False,False,False
2,CUST00001,480000,1614000,480000,70.26,4,0,0,0.297398,0.297398,157045.454545,322954.545455,False,False,False,False,False,True,False
3,CUST00001,52000,1614000,532000,67.04,5,1,0,0.032218,0.329616,157045.454545,-105045.454545,False,True,False,False,False,True,False
4,CUST00001,45000,1614000,45000,97.21,6,1,0,0.027881,0.027881,157045.454545,-112045.454545,False,False,False,False,True,True,False
